In [10]:
import pandas as pd
import numpy as np
import h5py
import datetime
from datetime import datetime
from datetime import timedelta 
import time
import json
import os

# Métodos

In [4]:
def delete_hdf_table(file_path, key):
    with pd.HDFStore(file_path, mode='a') as store:
        if f'/{key}' in store.keys():
            store.remove(f'/{key}')
            print(f"Table '{key}' removed from HDF5.")
        else:
            print(f"Table '{key}' not found in HDF5.")

def read_hdf(file_path, key):
    return pd.read_hdf(file_path, mode = 'r', key=key, encoding='utf-8')

def show_hdf_tables(file_path):
    with pd.HDFStore(file_path) as store:
        keys = store.keys()
        print(f"Current tables in {file_path}:")
        for key in keys:
            print(f"  - {key.lstrip('/')}")

def write_hdf_chunks(df, path, key, chunksize=10_000_000):
    with pd.HDFStore(path, mode='a', complevel=9, complib='blosc:lz4') as store: # Open in read+write mode to append
        for i in range(0, len(df), chunksize):
            chunk = df.iloc[i:i+chunksize]
            append_mode = i != 0 # Append after the first chunk to avoid overwriting data from previous chunks
            store.append(
                key,
                chunk,
                format='table',
                data_columns=True,
                min_itemsize={'gauge_code': 20},
                encoding='utf-8',
                append=append_mode
            )
            del chunk  # free memory
            print(f"Written rows {i} to {min(i+chunksize-1, len(df))}")

def read_hdf_chunks(file_path, key, chunksize=1_000_000):
    """
    Read HDF file in chunks and concatenate into a single DataFrame.
    Only works if the HDF key is stored in 'table' format.
    
    Parameters:
    file_path (str): Path to the HDF5 file
    key (str): Key/table name to read
    chunksize (int): Number of rows per chunk
    
    Returns:
    pd.DataFrame: Concatenated DataFrame
    """
    try:
        with pd.HDFStore(file_path, mode='r+') as store:  # Changed to 'r+' (read/write) and automatically closes the store
            if store.get_storer(key).is_table:
                dfs = []
                i = 1
                for chunk in store.select(key, chunksize=chunksize):
                    dfs.append(chunk)
                    print(f"Chunk {i} with {len(chunk)} rows read.")
                    i += 1                
                if dfs:  # Check if any chunks were read
                    return pd.concat(dfs, ignore_index=True)
                else:
                    print("No data found or chunksize too large.")
                    return pd.DataFrame()
            else:
                # If not table format, read all at once
                print("Not in table format, reading all data at once.")
                return store.select(key)  # Convert to DataFrame
    except FileNotFoundError:
        print(f"File {file_path} not found.")
        return pd.DataFrame()
    except KeyError:
        print(f"Key {key} not found in file {file_path}.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading HDF file: {e}")
        return pd.DataFrame()


# Parâmetros

In [ ]:
cleaned_path = './1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5'
# upper_threshold = 450  # mm/day
# lower_threshold = 0  # mm/day
# rainfall_threshold = 200.0  # mm

# parameters_path = './parameters.json'
# if os.path.exists(parameters_path):
#     with open(parameters_path, 'r') as f:
#         params = json.load(f)
#         lower_threshold =params["thresholds"].get("lower", "No 'lower' key found")
#         upper_threshold = params["thresholds"].get("upper", "No 'upper' key found")
#         rainfall_threshold = params["thresholds"].get("extreme_event", "No 'extreme_event' key found")
#         print("Parameters loaded successfully.")
#         print(f"Lower threshold: {lower_threshold}")
#         print(f"Upper threshold: {upper_threshold}")
#         print(f"Rainfall threshold: {rainfall_threshold}")
# else:
#     print(f"Parameters file {parameters_path} not found.")
#     params = {}

Parameters loaded successfully.
Lower threshold: 0.0
Upper threshold: 450.0
Rainfall threshold: 200.0


# Data processing

In [6]:
show_hdf_tables(cleaned_path)

Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_preclassif


In [8]:
chunk_size = 10_000_000  # Adjust the chunk size as needed
df_data = read_hdf_chunks(cleaned_path, key='table_data_outlier_filtered', chunksize=chunk_size)
df_data

Chunk 1 with 10000000 rows read.
Chunk 2 with 10000000 rows read.
Chunk 3 with 10000000 rows read.
Chunk 4 with 10000000 rows read.
Chunk 5 with 10000000 rows read.
Chunk 6 with 10000000 rows read.
Chunk 7 with 10000000 rows read.
Chunk 8 with 10000000 rows read.
Chunk 9 with 10000000 rows read.
Chunk 10 with 10000000 rows read.
Chunk 11 with 10000000 rows read.
Chunk 12 with 10000000 rows read.
Chunk 13 with 3590257 rows read.


,gauge_code,datetime,rain_mm
0,110018901A,2021-02-02,9.48
1,110018901A,2021-02-03,4.73
2,110018901A,2021-02-04,73.28
3,110018901A,2021-02-07,0.20
4,110018901A,2021-02-08,0.00
...,...,...,...
123590252,88690050,2023-12-27,0.00
123590253,88690050,2023-12-28,0.00
123590254,88690050,2023-12-29,2.00
123590255,88690050,2023-12-30,0.00


In [22]:
df_data['year'] = pd.to_datetime(df_data['datetime']).dt.year
df_p_availability = df_data.groupby(['gauge_code', 'year']).agg({'rain_mm': 'count'}).reset_index()
df_p_availability['days_in_year'] = df_p_availability.apply(lambda x: 365 if (x['year'] % 4 != 0 or (x['year'] % 100 == 0 and x['year'] % 400 != 0)) else 366, axis=1)
df_p_availability['p_availability'] = df_p_availability['rain_mm'] / df_p_availability['days_in_year'] * 100
df_p_availability['p_availability'] = df_p_availability['p_availability'].fillna(0).replace([np.inf, -np.inf], 0)
df_p_availability = pd.DataFrame(df_p_availability[['gauge_code', 'year', 'p_availability']])
df_p_availability

,gauge_code,year,p_availability
0,00047000,1961,100.00000
1,00047000,1962,100.00000
2,00047000,1963,100.00000
3,00047000,1964,100.00000
4,00047002,1977,6.30137
...,...,...,...
345863,S713,2021,100.00000
345864,S714,2021,100.00000
345865,S715,2021,100.00000
345866,S716,2021,100.00000


In [ ]:
# df_p_availability.to_hdf(cleaned_path
#                          , key = 'table_p_availability'
#                          , encoding = 'utf-8'
#                          , mode='r+'
#                          , format='table'
#                          , complevel=9
#                          , append=False)

df_p_availability = write_hdf_chunks(df_p_availability
                         , cleaned_path
                         , key = 'table_p_availability'
                         , chunksize=100_000)
show_hdf_tables(cleaned_path)

Written rows 0 to 99999
Written rows 100000 to 199999
Written rows 200000 to 299999
Written rows 300000 to 345868
Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_p_availability
  - table_preclassif


In [27]:
df_p_availability = read_hdf(cleaned_path, key='table_p_availability')
df_p_availability

,gauge_code,year,p_availability
0,00047000,1961,100.00000
1,00047000,1962,100.00000
2,00047000,1963,100.00000
3,00047000,1964,100.00000
4,00047002,1977,6.30137
...,...,...,...
345863,S713,2021,100.00000
345864,S714,2021,100.00000
345865,S715,2021,100.00000
345866,S716,2021,100.00000
